# osu-ai-dev: BC новой линии (coords-вход) на бесплатном GPU Colab

Ноутбук гоняет `bc_train3.py` — BC с нуля на входе v3: к 8 игровым кадрам
добавлены координатные каналы (курсор, цель, кнопки btn_l/btn_r — стек 12),
учитель — `oracle_v4` (машина состояний кнопок: все 5 классов получают метки),
в данных телепорты (толчок / к цели / в случайную точку) и единый лаг
0/1/2 = 0.40/0.35/0.25.

**Почему новая линия**: пять прогонов старой (BC, А-кольцо, Б-чанки, В-плотность)
упираются в одно и то же — cl nav 66–68px при живом teacher-forced 3–7px;
far_probe показал, что направление «курсор → цель» из пикселей не извлекается,
а состояние кнопок в кадре отсутствовало. Домой забираешь **два файла весов**.

## Что приготовить заранее (2 файла)

1. **`osu-ai-dev-stage4.zip`** — код новой линии (ноутбук проверит: внутри
   `bc_train3.py` + `coords.py` + `oracle_v4.py`).
2. **`sim_maps.zip`** — твои карты (те же, что и раньше). В PowerShell из
   корня osu-ai-dev:

   ```
   Compress-Archive -Path sim_maps, sim_maps_t1 -DestinationPath sim_maps.zip
   ```

   (внутри архива должны быть папки `sim_maps/` и `sim_maps_t1/` с .osu-файлами)

## Ориентиры времени на T4

- короткий тест (ячейка 4): ~2–3 мин;
- основной прогон 250k шагов + прогрев 75k: **~1.5–2 ч** в fp32 (вход стал
  12 каналов вместо 8 — чуть тяжелее; кривая старой линии выходила на плато
  после ~150k). Полный вариант 500k/150k (дефолт скрипта) ≈ вдвое дольше.

## Жёлтые предупреждения в начале — это НЕ ошибки

- «Gym has been unmaintained since 2022…» — прощальное письмо старого пакета
  `gym`, который лежит в образе Colab с давних времён и в нашем конвейере не
  используется: код проекта и sb3 работают только с `gymnasium` (он как раз
  установлен). Игнорируй.
- «absl … cudart_stub.cc … GPU will not be used» — TensorFlow из образа Colab
  сообщает, что не нашёл драйверов ДЛЯ СЕБЯ. Наш процесс GPU прекрасно видит
  (`device: cuda` + буфер кадров в VRAM) — на прогон не влияет.
- «jupyter_client … datetime.utcnow() … deprecated» — внутренности самих
  ноутбуков Colab, к обучению отношения не имеет.

Ошибка — это красный блок с Traceback. Всё жёлтое — шум.

Не закрывай вкладку во время основного прогона. Если Colab отключится —
перезапусти с ячейки 1 (данные теряются, веса пишутся только в конце).


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "Нет GPU: меню 'Среда выполнения' -> 'Сменить среду выполнения' -> T4 GPU, затем снова")
p = torch.cuda.get_device_properties(0)
free_b, total_b = torch.cuda.mem_get_info()
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {p.name} | VRAM {total_b / 2**30:.1f} ГБ, свободно {free_b / 2**30:.1f} ГБ")
print(f"compute capability sm_{cap[0]}{cap[1]} → железный bf16: {cap[0] >= 8}")
print("(T4 = sm_75: bf16 там только эмуляция без ускорения — код сам")
print(" оставит fp32 + буфер в VRAM, это норма)")

In [ ]:
import glob
import os
import shutil
import zipfile

from google.colab import files

print("Выбери ДВА архива: osu-ai-dev-stage4.zip и sim_maps.zip")
up = files.upload()
os.makedirs("/content/up", exist_ok=True)
for name, blob in up.items():
    open(f"/content/up/{name}", "wb").write(blob)

os.makedirs("/content/osu-ai-dev", exist_ok=True)
for name in up:
    tmp = f"/content/up/x_{name}"
    os.makedirs(tmp, exist_ok=True)
    with zipfile.ZipFile(f"/content/up/{name}") as z:
        z.extractall(tmp)
    # это архив проекта? (внутри есть bc_train3.py — новая линия)
    hits = glob.glob(f"{tmp}/**/bc_train3.py", recursive=True)
    if hits:
        src = os.path.dirname(hits[0])
        for f in os.listdir(src):
            dst = os.path.join("/content/osu-ai-dev", f)
            if os.path.exists(dst) and os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.move(os.path.join(src, f), dst)
        print(f"[ok] {name}: код проекта распакован")
        continue
    # иначе ищем папки карт
    found = False
    for d in ("sim_maps", "sim_maps_t1"):
        for h in [p for p in glob.glob(f"{tmp}/**/{d}", recursive=True) if os.path.isdir(p)]:
            dst = f"/content/osu-ai-dev/{d}"
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.move(h, dst)
            print(f"[ok] {name}: {d} — {len(glob.glob(dst + '/*.osu'))} карт")
            found = True
    if not found:
        # россыпь .osu прямо в корне архива без папок
        loose = glob.glob(f"{tmp}/*.osu") + glob.glob(f"{tmp}/**/*.osu", recursive=True)
        if loose and not glob.glob("/content/osu-ai-dev/sim_maps/*.osu"):
            os.makedirs("/content/osu-ai-dev/sim_maps", exist_ok=True)
            for f in loose:
                shutil.move(f, "/content/osu-ai-dev/sim_maps/")
            print(f"[ok] {name}: {len(loose)} .osu положены в sim_maps/")

# проверки: новая линия целиком (bc_train3 + coords + oracle_v4)
assert os.path.exists("/content/osu-ai-dev/bc_train3.py"), (
    "СТАРЫЙ архив (bc_train3.py нет — в нём линия bc_train.py) — нужен osu-ai-dev-stage4.zip")
src3 = open("/content/osu-ai-dev/bc_train3.py").read()
assert "oracle_v4" in src3, "bc_train3.py без oracle_v4 — странный архив, пришли мне его"
for f in ("coords.py", "oracle_v4.py", "far_probe.py", "dagger_gap.py",
          "diagnose_v3.py", "verify_transfer.py"):
    assert os.path.exists(f"/content/osu-ai-dev/{f}"), f"в архиве нет {f}"
n_maps = (len(glob.glob("/content/osu-ai-dev/sim_maps/*.osu"))
          + len(glob.glob("/content/osu-ai-dev/sim_maps_t1/*.osu")))
print("\nкарт всего:", n_maps)
assert n_maps > 0, "нет .osu карт — проверь sim_maps.zip (папки sim_maps/ и sim_maps_t1/)"
print("готово. дальше — зависимости")

In [ ]:
# ВАЖНО: НЕ ставь requirements.txt целиком в Colab (torch трогать нельзя —
# сломается GPU-сборка Colab). Ставим только то, чего не хватает.
!pip install -q "stable-baselines3==2.9.2"

import cv2
import gymnasium
import numpy
import stable_baselines3

print("sb3", stable_baselines3.__version__, "| cv2", cv2.__version__,
      "| gymnasium", gymnasium.__version__, "| numpy", numpy.__version__)

## Ячейка 4 — короткий тест конвейера (~2–3 мин)

Проверяет весь путь новой линии (сбор с coords-каналами -> BC -> перенос в
PPO -> самопроверка на реальных состояниях) на масштабе 5000 шагов. Ждём в
логе: `[stack] N_STACK=12 (coords=full) | … учитель: oracle_v4`, в конце —
«перенос чист». Если тут всё чисто — гоняем основной прогон.

In [ ]:
%cd /content/osu-ai-dev
!python bc_train3.py --steps 5000 --critic-warmup 0 --out bc3_colab_test

## Ячейка 5 — основной прогон

250k шагов + прогрев критика 75k. Полный вариант — без `--steps/--critic-warmup`
(500k/150k, вдвое дольше). Ждать «сохранено: bc3_policy.zip +
bc3_policy_bcnet.pt» и «перенос чист». На T4 в логе НЕ должно быть строки
`[amp]` — код сам держит fp32 (появилась — архив старый, скорость будет x2
хуже). Имя bc3_policy старую bc_policy_fresh НЕ затирает.

In [ ]:
%cd /content/osu-ai-dev
!python bc_train3.py --steps 250000 --critic-warmup 75000 --out bc3_policy

In [ ]:
%cd /content/osu-ai-dev
# проверке GPU не нужен; прячем CUDA — стек 12 читается из самих весов
!CUDA_VISIBLE_DEVICES="" python verify_transfer.py bc3_policy

In [ ]:
from google.colab import files

%cd /content/osu-ai-dev
files.download("bc3_policy.zip")
files.download("bc3_policy_bcnet.pt")

## Дальше на своём компе

Веса (`bc3_policy.zip` + `bc3_policy_bcnet.pt`) — это просто файлы тензоров,
не привязаны ни к GPU, ни к ОС, ни к Colab. Положи оба в корень `osu-ai-dev`.

**Первый замер — главный гейт новой линии** (все три инструмента понимают
coords-стек и .pt напрямую, запускаются на CPU, не мешают обучению):

```
python far_probe.py bc3_policy_bcnet.pt --lag-p 0.4,0.35,0.25
```

по факту первого прогона (2026-11-08): xfar 0г / bin 0.52 / aux 14г (было
45г / 0.35 / aux мертва 42-93г) — стена «далеко → поверни» сломана. Ориентир
для повторных прогонов: xfar bin ≥ 0.5 и углы ≤ 30г; решающий замер — cl nav
в dagger_gap ≤ 25px (факт: 9px против 66–68px у всей старой линии).

```
python dagger_gap.py bc3_policy_bcnet.pt --maps sim_maps --steps 4000 --lag-p 0.4,0.35,0.25
python diagnose_v3.py bc3_policy_bcnet.pt sim_maps 0 30 det rand --lag-p 0.4,0.35,0.25
```

в dagger_gap смотри строку «btn recall 0..4»: класс 1 (нажать Л) — это
клик-триггер, главный кандидат в ботлненек после слома стены прицела
(воронка diagnose: 31 объект → над объектом 13 → кликов 8 → хитов 6).

**DAgger-итерации (линия bc3, дома на GPU ~3–4.5 ч):** dagger_train2 обновлён
под coords-стек + учитель oracle_v4. Дефолт --teacher v4 — правильный для
bc3; для весов СТАРОЙ линии (bc_policy*, dagger2* до bc3) добавь --teacher v3,
иначе btn-метки не совпадут (agree btn будет артефактом).

```
python dagger_train2.py --iter 1 --resume-bc bc3_policy_bcnet.pt
```

дефолты: 300k шагов, oracle-share 0.33, чанк 20k x 32 эпохи, пробник после
каждого чанка; live-чекпоинт dagger2_iter1_live.pt — замер без остановки
обучения: `python live2zip.py dagger2_iter1_live.pt` → dagger_gap по .zip.
iter2: `--resume-bc dagger2_iter1_bcnet.pt --oracle-share 0.2`.

Если нужен A/B-контроль «старый вход, новый учитель»:

```
python bc_train3.py --coords none --out bc3_ctrl
```

(8 каналов — совместим со всей старой линией; DAgger по нему — с --teacher v4,
это контроль bc3_ctrl, не старые веса).

Нюансы:

- **Версии**: Colab живёт на своём torch (его не трогаем), sb3 ставим 2.9.2 —
  как в тестах. Если дома `PPO.load` вдруг ругнётся на файл — пришли текст
  ошибки, лечится `pip install stable-baselines3==2.9.2`.
- **verify_transfer дома**: свежий скрипт из архива работает сам; если
  запускаешь старую копию и падает на «FloatTensor» — PowerShell:
  `$env:CUDA_VISIBLE_DEVICES=''; python verify_transfer.py bc3_policy`.
- **Если Colab отключился** во время прогона — ничего не сохранилось (веса
  пишутся в самом конце): перезапусти ноутбук с ячейки 1.
- **Кривые**: move acc новой линии напрямую со старой НЕ сравнивай — вход
  другой (12 каналов + другие метки кнопок). Сравнение через замеры
  far_probe / dagger_gap / diagnose_v3, не по train acc.
- **Ещё не coords-совместимы**: train_stage (PPO) и play — обновим после
  положительной DAgger-итерации.
